<a href="https://colab.research.google.com/github/sergkurilenko/DeepML/blob/HW2/DeepML_HW2_Kurilenko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Куриленко Сергей Глубокое обучение HW2

Импорт библиотек

In [35]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

1. Загрузка данных

In [30]:

X = np.load('images.npy')
y = np.load('labels.npy')
X_test = np.load('images_sub.npy')

2. Нормализация

In [31]:
X = X.astype('float32') / 255.
X_test = X_test.astype('float32') / 255.

3. Train/Val split

In [32]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

3. Аугментации

In [33]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
datagen.fit(X_train)

4. Создаем модель

In [34]:
base_model = EfficientNetB0(
    include_top=False,
    weights=None,
    input_shape=(48, 48, 3)
)

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(26, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [36]:
# Callbacks
early_stop = EarlyStopping(patience=12, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(patience=4, factor=0.5, verbose=1)
checkpoint = ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy', verbose=1)

5. Обучение

In [37]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    validation_data=(X_val, y_val),
    epochs=80,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 1/80


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.0456 - loss: 3.3099
Epoch 1: val_accuracy improved from -inf to 0.05050, saving model to best_model.keras
282/282 ━━━━━━━━━━━━━━━━━━━━ 163s 272ms/step - accuracy: 0.0456 - loss: 3.3097 - val_accuracy: 0.0505 - val_loss: 3.2527 - learning_rate: 0.0010
Epoch 2/80
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.0471 - loss: 3.2296
Epoch 2: val_accuracy did not improve from 0.05050
282/282 ━━━━━━━━━━━━━━━━━━━━ 16s 58ms/step - accuracy: 0.0471 - loss: 3.2296 - val_accuracy: 0.0440 - val_loss: 3.2600 - learning_rate: 0.0010
Epoch 3/80
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.0597 - loss: 3.1808
Epoch 3: val_accuracy improved from 0.05050 to 0.07000, saving model to best_model.keras
282/282 ━━━━━━━━━━━━━━━━━━━━ 17s 60ms/step - accuracy: 0.0598 - loss: 3.1807 - val_accuracy: 0.0700 - val_loss: 3.2877 - learning_rate: 0.0010
Epoch 4/80
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.0792 - loss: 3.0792
Epoch

6. Валидационная метрика

In [38]:
val_pred = model.predict(X_val).argmax(axis=1)
print("Validation accuracy:", np.mean(val_pred == y_val))

63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 105ms/step
Validation accuracy: 0.9105


7. Предсказания

In [39]:
y_pred = model.predict(X_test, batch_size=128)
y_pred_labels = np.argmax(y_pred, axis=1)

391/391 ━━━━━━━━━━━━━━━━━━━━ 26s 37ms/step


8. Выгрузка

In [40]:
submission = pd.DataFrame({'Id': np.arange(len(y_pred_labels)), 'Category': y_pred_labels})
submission.to_csv('submission.csv', index=False)